In [0]:
# Cell 1: parameters
dbutils.widgets.text("catalog", "dev")
dbutils.widgets.text("schema", "bikeshare")
dbutils.widgets.text("volume", "raw")
dbutils.widgets.text("run_date", "")         # empty = today (UTC); job passes {{job.trigger.time.iso_datetime}}
dbutils.widgets.text("lookback_days", "2")
dbutils.widgets.text("forecast_days", "3")   # counts run_date as day 1
dbutils.widgets.text("locations", "all")     # "all" or "lat,lon"

catalog, schema, volume = (dbutils.widgets.get(k) for k in ("catalog", "schema", "volume"))
VOL = f"/Volumes/{catalog}/{schema}/{volume}"
print(VOL)

In [0]:
# Cell 2: time window and API choice
from datetime import date, datetime, timedelta, timezone

rd = dbutils.widgets.get("run_date")
run_date = date.fromisoformat(rd[:10]) if rd else datetime.now(timezone.utc).date()   # [:10] keeps 'YYYY-MM-DD'
start = run_date - timedelta(days=int(dbutils.widgets.get("lookback_days")))
end   = run_date + timedelta(days=int(dbutils.widgets.get("forecast_days")) - 1)

FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
HIST_URL     = "https://historical-forecast-api.open-meteo.com/v1/forecast"
today = datetime.now(timezone.utc).date()
url = HIST_URL if end < today else FORECAST_URL
print(run_date, start, end, url)

In [0]:
# Cell 3: locations
loc = dbutils.widgets.get("locations").strip()
if loc == "all":
    rows = spark.table(f"{catalog}.{schema}.silver_weather_locations").select("location_id", "lat", "lon").collect()
    locs = [(r.location_id, float(r.lat), float(r.lon)) for r in rows]
else:
    lat, lon = (float(x) for x in loc.split(","))
    locs = [(f"{lat}_{lon}", lat, lon)]
print(len(locs), "locations")

In [0]:
# Cell 4: request (reuse the retry helper from pull_gbfs, with a params argument)
import time, json, requests

MODEL = "meteoswiss_icon_seamless"
VARS  = "temperature_2m,relative_humidity_2m,precipitation,snowfall,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m,sunshine_duration"

def get_json(url, params, attempts=3, wait_s=5):
    for i in range(attempts):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 400:
                raise ValueError(f"Open-Meteo rejected the request: {r.text[:300]}")
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.RequestException, ValueError) as e:
            print(f"Attempt {i + 1}/{attempts} failed for {url}: {e}")
            if i == attempts - 1:
                raise
            time.sleep(wait_s * (i + 1))

params = {
    "latitude":  ",".join(str(l[1]) for l in locs),
    "longitude": ",".join(str(l[2]) for l in locs),
    "hourly": VARS,
    "models": MODEL,
    "start_date": start.isoformat(),
    "end_date":   end.isoformat(),
    "timeformat": "unixtime",
}
resp = get_json(url, params)
if isinstance(resp, dict):          # one location → dict, several → list
    resp = [resp]
assert len(resp) == len(locs), (len(resp), len(locs))

In [0]:
# Cell 5: write ONE JSON Lines file
pulled_at = datetime.now(timezone.utc)
folder = f"{VOL}/weather"
dbutils.fs.mkdirs(folder)
path = f"{folder}/weather_{start:%Y%m%d}_{end:%Y%m%d}_pulled_{pulled_at:%Y%m%dT%H%M%SZ}.json"

with open(path, "w") as fh:
    for (location_id, lat, lon), r in zip(locs, resp):
        record = {
            **r,                        # Open-Meteo response: latitude, longitude, elevation, 
            "location_id": location_id, # ...so our fields overwrite any key with the same name
            "requested_lat": lat,
            "requested_lon": lon,
            "pulled_at": pulled_at.isoformat(),   # ISO string
            "run_date": run_date.isoformat(),     # optional: which window this run was for
            "source_url": url,
            "model": MODEL
        }
        fh.write(json.dumps(record) + "\n")
print(len(resp), "locations ->", path)

In [0]:
resp